# Phase 2 · Legal NER · Deadline night · W3b · 2 more LoRMA seeds on all 70 documents (test probabilities only)
Trains LoRMA batch 4 with seeds (303, 404) on all 70 documents and saves `test_scores_W3b.pt` for notebook **W4**. Nothing to submit from here. About 38 minutes.

**Pipeline (identical in all 3 Phase-2 notebooks; only cell 1 differs):**
1. Text → **words with exact character offsets** (punctuation split off words; same BPE vocabulary) → BPE tokens.
2. Each window = **128-token document header** (case title, parties, counsel) + separator + the next ~900 tokens,
   so a name deep in the judgment can still be linked to "Appellant"/"Respondent". Header words are never labelled.
3. Phase-1 backbone (causal mask removed) → hidden state of the **first token of each word** → head → word-level BIO tags
   (a span can never start or end inside a word).
4. Overlapping windows; each word's prediction is the tent-weighted average over the windows that contain it.
5. Decoding chosen **on validation**: greedy vs BIO-constrained Viterbi, with/without document-level label propagation.
6. Strict span scoring (exact start, end, label; macro over 12 labels) on 10 held-out training documents, then the test fold.

**Order inside this notebook:** (a) a **smoke check** runs every step for 1 epoch on 8 documents (~1–2 min per step) and
stops on the first error, (b) the real runs. Total about 38 min.

**Run:** GPU **T4 x2**, Internet **On**, your Hugging Face secret (`capstone` or `HF_TOKEN`), Add Input → **`phase2-backbones`**.
**Save & Run All.** Outputs per run in `/kaggle/working/<run>/`: `submission_<run>.zip`, `record.json`,
`trainable_weights.pt`. **Never submit a `smoke_*` zip.**

In [ ]:
# 1 · Settings (the only cell that differs between the Phase-2 notebooks)
import os, sys, json, glob, time, random, math, gc
import numpy as np, torch
BASE = dict(
    arch="arch4_mla", method="full_ft", head="linear", bidirectional=True,
    split="fine",                 # punctuation split off words (offset ceiling 0.995 vs 0.73 for "bpe")
    window=1024, header_tokens=128,   # 1024-token windows incl. a 128-token document header (case title, parties)
    batch_size=8, accum=1, eval_batch=16, epochs=12, patience=3,
    lr=3e-5, lr_head=1e-3, weight_decay=0.01, warmup_frac=0.1, clip=1.0, dropout=0.2,
    class_weighting="none", rank=16, alpha=32, bottleneck=64,
    seed=1337, max_train_hours=2.5, log_every=50, save_full=False,
)
TAG = "W3b"
PLAN = [dict(cfg=dict(BASE, method="lorma", rank=16, alpha=32, lr=5e-4, batch_size=4, epochs=14, run=f"p2_w3b_lorma_s{s}_all70", seed=s, refit_all=True, decoding="viterbi", keep_scores=True)) for s in (303, 404)]
OUT = "/kaggle/working"
SMOKE_CHECK = True         # run every step for 1 epoch on 8 documents first (stops on the first error)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
found = glob.glob("/kaggle/input/**/phase2-backbones/manifest.json", recursive=True) or \
        glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert found, "attach the dataset phase2-backbones (Add Input -> Your Datasets)"
ROOT = os.path.dirname(found[0])
print("backbones:", ROOT, "| GPUs:", torch.cuda.device_count(), "| steps:", [s["cfg"]["run"] for s in PLAN])

In [ ]:
# 2 · Hugging Face token (the dataset is gated). Tries the Kaggle secrets "capstone" and "HF_TOKEN".
from huggingface_hub import login, whoami
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for _name in ("capstone", "HF_TOKEN"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(_name)
            if HF_TOKEN: break
        except Exception:
            pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "no Hugging Face token: add it as a Kaggle secret (Add-ons -> Secrets) and attach it to this notebook"
login(token=HF_TOKEN)
print("Hugging Face user:", whoami(token=HF_TOKEN)["name"])

## 3 · Phase-2 code
`ner_data.py` (pure Python): parsing, words with character offsets, BIO/span targets, decoding, propagation, strict scorer, error analysis. `ner_model.py`: backbones (4 architectures), adaptation methods (full, partial, BitFit, LoRA, LoRMA, Houlsby/Pfeiffer adapters, probe), heads (linear, CRF, span), distillation, training, prediction. `ner_run.py`: one complete experiment / a plan of experiments.

In [ ]:
os.makedirs('p2src', exist_ok=True)

In [ ]:
%%writefile p2src/ner_data.py
# Phase 2 (L-NER) data utilities (pure Python, no torch):
#   gold parsing, words with exact character offsets, word-level BIO tags and span targets,
#   span decoding strategies, document-level label propagation, strict scoring, error analysis, submission.
import bisect
import re
from collections import Counter, defaultdict

# ======================================================================================================
# 1. Gold annotations
# ======================================================================================================
_START = ("start", "start_char", "start_offset", "begin", "s")
_END = ("end", "end_char", "end_offset", "stop", "e")
_LABEL = ("label", "type", "entity_type", "tag", "category")
_TEXT = ("text", "entity_text", "span", "mention", "value")
_IDS = ("id", "doc_id", "document_id", "docid")
_ENTS = ("entities", "annotations", "spans", "labels", "gold")


def _pick(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    return None


def _entity(item):
    """one annotation in any common shape -> (start, end, label, text or None)"""
    if isinstance(item, (list, tuple)):
        s, e, lab = item[0], item[1], item[2]
        txt = item[3] if len(item) > 3 else None
    elif isinstance(item, dict):
        s, e, lab, txt = _pick(item, _START), _pick(item, _END), _pick(item, _LABEL), _pick(item, _TEXT)
    else:
        raise ValueError(f"unknown annotation format: {item!r}")
    return int(s), int(e), str(lab), (txt if isinstance(txt, str) else None)


def parse_gold(raw):
    """gold json (dict doc -> list, dict doc -> {entities: [...]}, or list of records) -> {doc: [(s,e,lab,txt)]}"""
    out = {}
    if isinstance(raw, list):
        for rec in raw:
            out[str(_pick(rec, _IDS))] = [_entity(x) for x in (_pick(rec, _ENTS) or [])]
        return out
    for doc, v in raw.items():
        if isinstance(v, dict):
            v = _pick(v, _ENTS) or []
        out[str(doc)] = [_entity(x) for x in v]
    return out


def parse_texts(raw):
    """text json -> {doc: text}"""
    if isinstance(raw, list):
        return {str(_pick(r, _IDS)): _pick(r, ("text", "content", "document")) for r in raw}
    return {str(k): (v if isinstance(v, str) else _pick(v, ("text", "content", "document"))) for k, v in raw.items()}


def end_convention(texts, gold):
    """Is `end` exclusive (text[s:e] == entity text) or inclusive (text[s:e+1])? Decided by the data."""
    exc = inc = total = 0
    bad = []
    for doc, ents in gold.items():
        t = texts.get(doc)
        if t is None:
            continue
        for s, e, lab, txt in ents:
            if txt is None:
                continue
            total += 1
            if t[s:e] == txt:
                exc += 1
            elif t[s:e + 1] == txt:
                inc += 1
            elif len(bad) < 5:
                bad.append((doc, s, e, lab, txt, t[s:e + 1]))
    conv = "inclusive" if inc > exc else "exclusive"
    return conv, dict(total_with_text=total, exclusive_match=exc, inclusive_match=inc, examples_no_match=bad)


def to_exclusive(gold, conv):
    """-> {doc: [(s, e_exclusive, label)]} (internal convention: python slice)"""
    add = 1 if conv == "inclusive" else 0
    return {d: [(s, e + add, lab) for s, e, lab, _ in ents] for d, ents in gold.items()}


# ======================================================================================================
# 2. Tokens and words with character offsets
# ======================================================================================================
# The pre-tokenisation decides where an entity can start or end. "bpe" = the split used in pretraining
# (a word keeps its punctuation: "Singh," is one piece). "fine" = punctuation is split off words, so
# "Singh" | "," can be cut exactly. Every piece is still encoded by the same BPE (same vocabulary).
SPLITS = {
    ("space", "bpe"): re.compile(r" ?\S+|\s+(?!\S)|\s+"),
    ("space", "fine"): re.compile(r" ?[^\W_]+| ?_+| ?[^\w\s]|\s+(?!\S)|\s+"),
    ("a1", "bpe"): re.compile(r"\S+|\s+"),
    ("a1", "fine"): re.compile(r"[^\W_]+|_+|[^\w\s]|\s+"),
}


def tokenizer_kind(tok):
    return "space" if getattr(tok, "space_prefix", False) else "a1"


def encode_words(tok, text, split="fine"):
    """text -> ids, wtok, wcs, wce
    ids  : BPE token ids of the whole text (decode(ids) == text)
    words: every pre-token piece that is not pure whitespace. For word w:
           wtok[w] = index of its first non-whitespace token (the token that represents the word),
           [wcs[w], wce[w]) = its characters without surrounding whitespace."""
    pat = SPLITS[(tokenizer_kind(tok), split)]
    ids, wtok, wcs, wce = [], [], [], []
    covered = 0
    for m in pat.finditer(text):
        chunk, base = m.group(), m.start()
        assert base == covered, f"pre-tokeniser skipped characters at {covered}"
        covered = m.end()
        char_of_byte = []
        for ci, ch in enumerate(chunk):
            char_of_byte.extend([ci] * len(ch.encode("utf-8")))
        pos, first, ws, we = 0, None, None, None
        for t in tok.encode_chunk(chunk.encode("utf-8")):
            n = len(tok.itos[t])
            s, e = base + char_of_byte[pos], base + char_of_byte[pos + n - 1] + 1
            pos += n
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if s < e:
                if first is None:
                    first, ws = len(ids), s
                we = e
            ids.append(t)
        if first is not None:
            wtok.append(first); wcs.append(ws); wce.append(we)
    assert covered == len(text), "pre-tokeniser did not cover the whole text"
    return ids, wtok, wcs, wce


# ======================================================================================================
# 3. Targets: BIO tags per word, and (start word, end word, label) for the span head
# ======================================================================================================
IGNORE = -100


def tag_vocab(labels):
    tags = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
    return tags, {t: i for i, t in enumerate(tags)}


def word_targets(wcs, wce, spans, tag2id, label2id):
    """gold char spans -> (BIO tag per word, [(first_word, last_word, label_id)], #overlapping spans skipped)"""
    n = len(wcs)
    tags, taken, wspans, conflicts = [0] * n, [False] * n, [], 0
    for s, e, lab in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = bisect.bisect_right(wce, s)              # first word ending after s
        hi = bisect.bisect_left(wcs, e, lo)           # first word starting at/after e
        if lo >= hi:
            continue
        if any(taken[lo:hi]):
            conflicts += 1
            continue
        for k in range(lo, hi):
            tags[k] = tag2id[("B-" if k == lo else "I-") + lab]
            taken[k] = True
        wspans.append((lo, hi - 1, label2id[lab]))
    return tags, wspans, conflicts


def allowed_transitions(tags):
    """BIO validity: allowed[i][j] = may tag j follow tag i; start_ok[j] = may a sequence start with j."""
    K = len(tags)
    ok = [[True] * K for _ in range(K)]
    start_ok = [not t.startswith("I-") for t in tags]
    for i, a in enumerate(tags):
        for j, b in enumerate(tags):
            if b.startswith("I-") and not (a != "O" and a[2:] == b[2:]):
                ok[i][j] = False                      # I-X only after B-X or I-X
    return ok, start_ok


# ======================================================================================================
# 4. Decoding
# ======================================================================================================
def spans_from_tags(tags, wcs, wce, id2tag):
    """word tags -> [(s, e, label)]. B-X starts a span; I-X continues a span of type X; an I-X after O or
    after another type also starts a span (lenient repair, used with greedy argmax decoding)."""
    out, cur = [], None
    for t, s, e in zip(tags, wcs, wce):
        tag = id2tag[t]
        if tag == "O":
            if cur: out.append(cur); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [s, e, lab]
        else:
            cur[1] = e
    if cur: out.append(cur)
    return [tuple(x) for x in out]


def spans_from_candidates(cands, wcs, wce, id2label):
    """span head: cands = {(i, j): (prob, label_id)} (label_id 0 = none) -> non-overlapping spans,
    highest probability first (greedy)."""
    used, out = set(), []
    for (i, j), (p, l) in sorted(cands.items(), key=lambda kv: -kv[1][0]):
        if l == 0 or any(k in used for k in range(i, j + 1)):
            continue
        used.update(range(i, j + 1))
        out.append((wcs[i], wce[j], id2label[l]))
    return sorted(out)


_WORDCH = re.compile(r"\w")


def propagate(spans, text, min_chars=4):
    """Document-level consistency: a string predicted as label L somewhere in the document is also
    labelled L at its other (unlabelled, whole-word) occurrences. Majority label per string."""
    votes = defaultdict(Counter)
    for s, e, l in spans:
        t = text[s:e]
        if len(t) >= min_chars and _WORDCH.search(t):
            votes[t][l] += 1
    covered = sorted((s, e) for s, e, _ in spans)
    starts = [s for s, _ in covered]

    def free(s, e):
        k = bisect.bisect_left(starts, e)
        return not any(cs < e and ce > s for cs, ce in covered[max(0, k - 50):k])

    out = list(spans)
    for t, c in votes.items():
        lab = c.most_common(1)[0][0]
        i = text.find(t)
        while i >= 0:
            j = i + len(t)
            boundary = (i == 0 or not text[i - 1].isalnum()) and (j == len(text) or not text[j].isalnum())
            if boundary and free(i, j):
                out.append((i, j, lab))
                k = bisect.bisect_left(starts, i); covered.insert(k, (i, j)); starts.insert(k, i)
            i = text.find(t, i + 1)
    return sorted(set(out))


def windows(n, size, stride):
    """window start positions covering [0, n); the last one is pulled back to end at n, so every window
    (except in documents shorter than size) has exactly `size` tokens - no padding needed."""
    if n <= size:
        return [0]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return starts


# ======================================================================================================
# 5. Strict scoring and error analysis
# ======================================================================================================
def strict_scores(pred, gold, labels):
    """pred/gold: {doc: [(s,e,label)]} -> macro/micro P/R/F1 (exact start, end and label) + per label."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        rows[l] = dict(P=P, R=R, F1=F, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    used = [l for l in labels if rows[l]["support"] or rows[l]["fp"]]
    mac = lambda k: sum(rows[l][k] for l in used) / max(1, len(used))
    T, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro = 2 * T / (2 * T + FP + FN) if T else 0.0
    return dict(macro_F1=mac("F1"), macro_P=mac("P"), macro_R=mac("R"), micro_F1=micro, per_label=rows)


def error_breakdown(pred, gold):
    """every mistake classified: wrong label on an exact span, boundary error (overlap, same label),
    boundary+label error, missed entity (no overlapping prediction), spurious prediction."""
    c = Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        c["correct"] += len(p & g)
        ps, used = sorted(p - g), set()
        for s, e, l in sorted(g - p):
            over = [x for x in ps if x[0] < e and x[1] > s]
            if not over:
                c["missed"] += 1; continue
            used.update(over)
            if any(o[0] == s and o[1] == e for o in over): c["wrong_label_exact_span"] += 1
            elif any(o[2] == l for o in over): c["boundary_error_same_label"] += 1
            else: c["boundary_and_label_error"] += 1
        c["spurious"] += len([x for x in ps if x not in used])
    return dict(c)


def label_confusion(pred, gold):
    """exact-span matches with a different label: Counter[(gold label, predicted label)]"""
    c = Counter()
    for doc in gold:
        pm = {(s, e): l for s, e, l in pred.get(doc, [])}
        for s, e, l in gold[doc]:
            if (s, e) in pm and pm[(s, e)] != l:
                c[(l, pm[(s, e)])] += 1
    return c


# ======================================================================================================
# 6. Submission
# ======================================================================================================
def to_submission(pred, conv, texts=None):
    """{doc: [(s, e_exclusive, label)]} -> {doc: [[s, e, label(, text)]]} in the dataset's own end convention"""
    sub = 1 if conv == "inclusive" else 0
    out = {}
    for doc, spans in pred.items():
        rows = []
        for s, e, l in sorted(spans):
            row = [int(s), int(e - sub), l]
            if texts is not None: row.append(texts[doc][s:e])
            rows.append(row)
        out[doc] = rows
    return out

In [ ]:
%%writefile p2src/ner_model.py
# Phase 2 (L-NER) model side:
#   Phase-1 backbones as feature extractors (all 4 architectures, bidirectional or causal),
#   fine-tuning / parameter-efficient methods (full, partial, BitFit, LoRA, LoRMA, Houlsby/Pfeiffer adapters,
#   linear probe), prediction heads (linear softmax, CRF, span classifier), knowledge distillation,
#   sliding-window training and document-level prediction.
import bisect
import math
import os
import pickle
import random
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_data as nd

ARCH_INFO = {   # folder -> (tokenizer file, longest sequence the backbone supports)
    "arch1_decoder": ("a1_bpe8000.pkl", 1024),
    "arch2_encdec": ("a1_bpe8000.pkl", 768),
    "arch3_prefixlm": ("space_bpe8000.pkl", 1024),
    "arch4_mla": ("space_bpe8000.pkl", 1024),
}
NEG = -1e4


def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available())


def _add_code_path(root):
    code = os.path.join(root, "code")
    if code not in sys.path:
        sys.path.insert(0, code)


def load_tokenizer(arch, root):
    _add_code_path(root)
    import a1_tokenizer, space_bpe  # noqa: F401  (classes needed to unpickle)
    return pickle.load(open(os.path.join(root, "tokenizers", ARCH_INFO[arch][0]), "rb"))


# ======================================================================================================
# 1. Backbones
# ======================================================================================================
def _mla_forward(self, x, cos, sin, cache=None):
    """MLA attention of architecture 4 without the causal mask (used when self.bidirectional is True)."""
    if cache is not None or not getattr(self, "bidirectional", False):
        return self._causal_forward(x, cos, sin, cache)
    from rope import apply_rope
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2)
    q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
    q_rope = apply_rope(q_rope, cos.to(q.dtype), sin.to(q.dtype))
    c_kv = self.kv_norm(self.kv_down(x))
    k_r = apply_rope(self.k_rope(x)[:, None], cos.to(q.dtype), sin.to(q.dtype))
    k_nope = self.k_up(c_kv).view(B, T, self.h, self.nope).transpose(1, 2)
    v = self.v_up(c_kv).view(B, T, self.h, self.vd).transpose(1, 2)
    q = torch.cat([q_nope, q_rope], dim=-1)
    k = torch.cat([k_nope, k_r.expand(B, self.h, T, self.rope)], dim=-1)
    out = F.scaled_dot_product_attention(q, k, v, is_causal=False, dropout_p=self.dropout if self.training else 0.0)
    return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), None


class Backbone(nn.Module):
    """Phase-1 language model reduced to a feature extractor: token ids (B,T) -> hidden states (B,T,d).
    bidirectional: drop the causal mask (arch 1, 4), use the whole window as prefix (arch 3); the arch-2
    encoder is bidirectional by design. keep_layers: keep only the first k blocks (distillation student).
    init="random": same architecture, random weights (control experiment for transfer from Phase 1)."""

    def __init__(self, arch, root, bidirectional=True, keep_layers=None, init="pretrained"):
        super().__init__()
        _add_code_path(root)
        self.arch, self.bidirectional = arch, bidirectional
        ck = torch.load(os.path.join(root, arch, "ckpt_best.pt"), map_location="cpu", weights_only=False)
        cfg = dict(ck["cfg"])
        if arch == "arch4_mla":
            from mla import MLAConfig, MLALM, MLAttention
            if not hasattr(MLAttention, "_causal_forward"):
                MLAttention._causal_forward = MLAttention.forward
                MLAttention.forward = _mla_forward
            m = MLALM(MLAConfig(**cfg))
        elif arch == "arch3_prefixlm":
            from prefixlm import PrefixLMConfig, PrefixLM
            m = PrefixLM(PrefixLMConfig(**cfg))
        elif arch == "arch2_encdec":
            from encdec import EncDecConfig, EncDecLM
            m = EncDecLM(EncDecConfig(**cfg))
        elif arch == "arch1_decoder":
            from model_arch import LMConfig, DecoderLM
            m = DecoderLM(LMConfig(**cfg))
        else:
            raise ValueError(arch)
        if init == "pretrained":
            missing, unexpected = m.load_state_dict(ck["model"], strict=False)
            assert not unexpected and all(x in ("rope", "pos_emb", "mask") or "rope" in x or "mask" in x
                                          or "pos_emb" in x for x in missing), (missing[:5], unexpected[:5])
        else:
            for mod in m.modules():
                if isinstance(mod, (nn.Linear, nn.Embedding)):
                    nn.init.normal_(mod.weight, std=0.02)
                    if getattr(mod, "bias", None) is not None: nn.init.zeros_(mod.bias)
        self.phase1_step, self.phase1_val = ck.get("step"), ck.get("val_loss")
        if arch == "arch2_encdec":                        # NER uses only the (bidirectional) encoder
            m.decoder = nn.ModuleList(); m.dec_norm = nn.Identity()
        if arch == "arch1_decoder":                       # A3 attention keeps its causal mask in a buffer
            for blk in m.blocks:
                blk.attn.causal_mask.fill_(True) if bidirectional else None
        if arch == "arch4_mla":
            for blk in m.blocks:
                blk.attn.bidirectional = bidirectional
        if keep_layers:
            blocks = self._blocks(m)
            del blocks[keep_layers:]
        m.head = None if arch != "arch2_encdec" else m.head   # LM head is not used (tied to the embedding)
        self.m, self.d, self.max_len = m, cfg["d_model"], ARCH_INFO[arch][1]

    @staticmethod
    def _blocks(m):
        return m.encoder if hasattr(m, "encoder") else m.blocks

    @property
    def blocks(self):
        return self._blocks(self.m)

    @property
    def final_norm(self):
        m = self.m
        return m.enc_norm if hasattr(m, "enc_norm") else (m.ln_f if hasattr(m, "ln_f") else m.norm)

    def forward(self, ids):
        m = self.m
        if self.arch == "arch4_mla":
            return m.hidden(ids)
        if self.arch == "arch3_prefixlm":
            return m.hidden(ids, prefix_len=ids.size(1) if self.bidirectional else 0)
        if self.arch == "arch2_encdec":
            return m.encode(ids)
        B, T = ids.shape                                  # arch 1: DecoderLM.forward without the LM head
        x = m.tok_emb(ids)
        if m.cfg.scale_emb:
            x = x * math.sqrt(m.cfg.d_model)
        if m.cfg.pos_emb_type == "sinusoidal":
            x = x + m.pos_emb[:T].unsqueeze(0)
        for blk in m.blocks:
            x = blk(x)
        return m.ln_f(x)


# ======================================================================================================
# 2. Fine-tuning / adaptation methods
# ======================================================================================================
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): W x + (alpha/r) * B A x, with W frozen, A random, B = 0 (starts at W)."""

    def __init__(self, base, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (low-rank MULTIPLICATIVE adaptation): W' = (I + (alpha/r) B A) W, with W frozen.
    The identity term gives the product full rank ("rank inflation" by identity), and B = 0 makes the
    adapted layer start exactly at the pretrained one. Computed as h + s * B(A h) with h = W x, so the
    d_out x d_in product (BA)W is never formed. A bias, if any, is added after the multiplicative update."""

    def __init__(self, base, r=16, alpha=32):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + self.scale * F.linear(F.linear(h, self.A), self.B)
        return h if self.base.bias is None else h + self.base.bias


class Adapter(nn.Module):
    """bottleneck adapter (Houlsby et al. 2019): x + up(GELU(down(x))), up = 0 at start (identity)."""

    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(x)))


class AdapterWrap(nn.Module):
    """applies an adapter to the output of a sub-layer (attention or feed-forward), before the residual add"""

    def __init__(self, module, d, bottleneck):
        super().__init__()
        self.module, self.adapter = module, Adapter(d, bottleneck)

    def __getattr__(self, name):                      # keep attributes of the wrapped module reachable
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["module"], name)

    def forward(self, *args, **kwargs):
        out = self.module(*args, **kwargs)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def _replace_linears(root_module, make):
    n = 0
    for name, child in list(root_module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(root_module, name, make(child)); n += 1
        else:
            n += _replace_linears(child, make)
    return n


def apply_method(net, cfg):
    """Freeze/unfreeze and insert adaptation modules according to cfg["method"]. The task head is always
    trained. Returns a short description for the record."""
    bb, method = net.backbone, cfg["method"]
    for p in bb.parameters():
        p.requires_grad = method == "full_ft"
    desc = method
    if method == "partial":                           # layer-wise: only the top k blocks + final norm
        k = cfg.get("train_top_layers", 4)
        for blk in list(bb.blocks)[-k:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in bb.final_norm.parameters(): p.requires_grad = True
        desc = f"partial (top {k} of {len(bb.blocks)} blocks)"
    elif method == "bitfit":                          # only bias terms (Ben Zaken et al. 2022)
        n = 0
        for name, p in bb.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True; n += p.numel()
        assert n > 0, "BitFit needs bias terms - this backbone has none (use arch 1 or arch 2)"
        desc = f"bitfit ({n:,} bias parameters)"
    elif method in ("lora", "lorma"):
        r, a = cfg.get("rank", 16), cfg.get("alpha", 32)
        make = (lambda lin: LoRALinear(lin, r, a, cfg.get("lora_dropout", 0.05))) if method == "lora" \
            else (lambda lin: LoRMALinear(lin, r, a))
        n = sum(_replace_linears(blk, make) for blk in bb.blocks)
        for mod in bb.modules():
            if isinstance(mod, (LoRALinear, LoRMALinear)):
                mod.A.requires_grad = True; mod.B.requires_grad = True
        desc = f"{method} (rank {r}, alpha {a}, {n} linear layers)"
    elif method in ("houlsby", "pfeiffer"):           # Houlsby: after attention and FFN; Pfeiffer: after FFN only
        bott = cfg.get("bottleneck", 64)
        for blk in bb.blocks:
            if method == "houlsby":
                blk.attn = AdapterWrap(blk.attn, bb.d, bott)
            blk.ffn = AdapterWrap(blk.ffn, bb.d, bott)
        for mod in bb.modules():
            if isinstance(mod, Adapter):
                for p in mod.parameters(): p.requires_grad = True
            if isinstance(mod, (nn.LayerNorm,)) or type(mod).__name__ == "RMSNorm":
                for p in mod.parameters(): p.requires_grad = True     # Houlsby also trains the norms
        desc = f"{method} adapters (bottleneck {bott})"
    elif method == "probe":                           # frozen backbone, only the head (linear probe)
        desc = "linear probe (frozen backbone)"
    elif method != "full_ft":
        raise ValueError(method)
    return desc


def count_params(net):
    total = sum(p.numel() for p in net.parameters())
    train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    return total, train


# ======================================================================================================
# 3. Prediction heads
# ======================================================================================================
class CRF(nn.Module):
    """linear-chain CRF over word tags. Learned transitions + fixed BIO constraints (forbidden moves = -1e4)."""

    def __init__(self, tags):
        super().__init__()
        K = len(tags)
        ok, start_ok = nd.allowed_transitions(tags)
        self.trans = nn.Parameter(torch.zeros(K, K)); self.start = nn.Parameter(torch.zeros(K))
        self.end = nn.Parameter(torch.zeros(K))
        self.register_buffer("ban", (~torch.tensor(ok)).float() * NEG)
        self.register_buffer("ban_start", (~torch.tensor(start_ok)).float() * NEG)

    def scores(self, train=False):
        # training windows may start inside an entity (first word I-X), so the start constraint is only
        # applied when decoding whole documents
        return self.trans + self.ban, (self.start if train else self.start + self.ban_start), self.end

    def nll(self, em, tags, mask):
        """em (B,W,K) float, tags (B,W) (pads ignored), mask (B,W) bool, prefix-contiguous"""
        trans, start, end = self.scores(train=True)
        B, W, K = em.shape
        tags = tags.clamp(min=0)
        mf = mask.float()
        # score of the gold path
        gold = start[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, W):
            step = trans[tags[:, t - 1], tags[:, t]] + em[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            gold = gold + step * mf[:, t]
        last = tags.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1)
        gold = gold + end[last]
        # log partition
        alpha = start[None] + em[:, 0]
        for t in range(1, W):
            nxt = torch.logsumexp(alpha[:, :, None] + trans[None], dim=1) + em[:, t]
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        logz = torch.logsumexp(alpha + end[None], dim=1)
        has = mask[:, 0]
        return ((logz - gold) * has.float()).sum() / has.float().sum().clamp(min=1)


def viterbi(em, trans, start, end):
    """em (n,K) -> best tag sequence (list) under transition scores (CPU)"""
    n, K = em.shape
    if n == 0:
        return []
    score = start + em[0]
    back = []
    for t in range(1, n):
        s = score[:, None] + trans
        best, idx = s.max(0)
        score = best + em[t]; back.append(idx)
    score = score + end
    tag = int(score.argmax())
    path = [tag]
    for idx in reversed(back):
        tag = int(idx[tag]); path.append(tag)
    return path[::-1]


class NERModel(nn.Module):
    """backbone -> word representations (first token of each word) -> head.
    head = "linear": softmax over BIO tags (cross-entropy, optional class weights)
           "crf":    BIO tag emissions + CRF (sequence-level likelihood)
           "span":   classify every word span up to max_span_words (start/end projections, bilinear term,
                     width embedding), label 0 = not an entity."""

    def __init__(self, backbone, tags, labels, head="linear", dropout=0.2, class_weights=None,
                 max_span_words=32, span_dim=256, neg_weight=0.5):
        super().__init__()
        self.backbone, self.head_type, self.tags = backbone, head, tags
        self.id2label = {i + 1: l for i, l in enumerate(labels)}          # span head: class 0 = no entity
        n_labels = len(labels)
        d, K = backbone.d, len(tags)
        self.drop = nn.Dropout(dropout)
        if head in ("linear", "crf"):
            self.head = nn.Linear(d, K)
            nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
            self.register_buffer("class_weights", class_weights if class_weights is not None else torch.ones(K))
            if head == "crf":
                self.crf = CRF(tags)
        elif head == "span":
            C = n_labels + 1
            self.maxw = max_span_words
            self.head = nn.ModuleDict(dict(
                s=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()), e=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()),
                ls=nn.Linear(span_dim, C), le=nn.Linear(span_dim, C), bil=nn.Linear(span_dim, C, bias=False),
                width=nn.Embedding(max_span_words, C)))
            w = torch.ones(C); w[0] = neg_weight
            self.register_buffer("class_weights", w)
        else:
            raise ValueError(head)

    def word_states(self, ids, wpos):
        h = self.backbone(ids)                                            # (B,T,d)
        return h.gather(1, wpos[:, :, None].expand(-1, -1, h.size(-1)))   # (B,W,d)

    def span_logits(self, hw, wmask):
        H = self.head
        s, e = H["s"](hw), H["e"](hw)                                     # (B,W,p)
        B, W, p = s.shape
        k = torch.arange(self.maxw, device=hw.device)
        j = torch.arange(W, device=hw.device)[:, None] + k[None]          # (W,maxw)
        valid = (j < W)[None] & wmask[:, :, None] & wmask.gather(1, j.clamp(max=W - 1).reshape(1, -1).expand(B, -1)).view(B, W, -1)
        ej = e.gather(1, j.clamp(max=W - 1).reshape(1, -1, 1).expand(B, -1, p)).view(B, W, self.maxw, p)
        logits = H["ls"](s)[:, :, None] + H["le"](ej) + H["bil"](s[:, :, None] * ej) + H["width"](k)[None, None]
        return logits.float(), valid

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        with autocast():
            hw = self.drop(self.word_states(ids, wpos))
            if self.head_type == "span":
                logits, valid = self.span_logits(hw, wmask)
            else:
                logits = self.head(hw).float()
        if self.head_type == "span":
            if stgt is None:                                   # invalid slots -> certainly "no entity"
                inv = ~valid
                out = logits.masked_fill(inv[..., None], NEG)
                out[..., 0] = out[..., 0].masked_fill(inv, 0.0)
                return out
            tgt = stgt.masked_fill(~valid, nd.IGNORE)
            return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                                   weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)
        if wtags is None:
            return logits
        if self.head_type == "crf":
            return self.crf.nll(logits, wtags, wmask).reshape(1)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), wtags.reshape(-1),
                               weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)


class KDWrapper(nn.Module):
    """Knowledge distillation (Hinton et al. 2015) at word level:
    loss = a * CE(student, gold) + (1 - a) * T^2 * KL(teacher_T || student_T). The teacher is frozen."""

    def __init__(self, student, teacher, T=2.0, alpha=0.5):
        super().__init__()
        self.student, self.teacher, self.T, self.alpha = student, teacher, T, alpha
        for p in teacher.parameters(): p.requires_grad = False

    def train(self, mode=True):
        super().train(mode); self.teacher.eval(); return self

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        if wtags is None:
            return self.student(ids, wpos, wmask)
        with torch.no_grad():
            t = self.teacher(ids, wpos, wmask)                            # (B,W,K)
        s = self.student(ids, wpos, wmask)
        m = wmask.reshape(-1)
        ce = F.cross_entropy(s.reshape(-1, s.size(-1)), wtags.reshape(-1), ignore_index=nd.IGNORE)
        sl, tl = s.reshape(-1, s.size(-1))[m] / self.T, t.reshape(-1, t.size(-1))[m] / self.T
        kl = F.kl_div(F.log_softmax(sl, -1), F.softmax(tl, -1), reduction="batchmean") * self.T ** 2
        return (self.alpha * ce + (1 - self.alpha) * kl).reshape(1)


# ======================================================================================================
# 4. Documents -> windows -> batches
# ======================================================================================================
def prepare_docs(texts, gold, tok, tag2id, label2id, split="fine"):
    docs, conflicts = {}, 0
    for d, t in texts.items():
        ids, wtok, wcs, wce = nd.encode_words(tok, t, split)
        r = dict(ids=ids, wtok=wtok, wcs=wcs, wce=wce)
        if gold is not None:
            r["wtags"], r["wspans"], c = nd.word_targets(wcs, wce, gold.get(d, []), tag2id, label2id); conflicts += c
        docs[d] = r
    return docs, conflicts


class Windowing:
    """Cuts every document into windows of `size` tokens. With header_tokens > 0, the first tokens of the
    document (case title, parties, counsel) are put in front of every window, followed by a separator,
    so each window can see who the appellant/respondent/counsel are. Header words are never labelled."""

    def __init__(self, tok, size, header_tokens=0, max_len=1024):
        self.sep = tok.encode("\n\n") if header_tokens else []
        self.H = header_tokens
        self.size = min(size, max_len - (header_tokens + len(self.sep) if header_tokens else 0))
        assert self.size >= 64, "window too small"

    def windows(self, docs, stride):
        out = []
        for d, r in docs.items():
            if r["wtok"]:
                for s in nd.windows(len(r["ids"]), self.size, stride):
                    out.append((d, s, min(self.size, len(r["ids"]))))
        return out

    def window_words(self, r, s, L):
        w0 = bisect.bisect_left(r["wtok"], s); w1 = bisect.bisect_left(r["wtok"], s + L)
        return w0, w1

    def batch(self, docs, b, device, head=None, maxw=None):
        ids, wpos, wtags, ranges, stgts = [], [], [], [], []
        for d, s, L in b:
            r = docs[d]
            head_ids = (r["ids"][:self.H] + self.sep) if self.H else []
            off = len(head_ids) - s
            ids.append(head_ids + r["ids"][s:s + L])
            w0, w1 = self.window_words(r, s, L)
            ranges.append((w0, w1))
            wpos.append([r["wtok"][w] + off for w in range(w0, w1)])
            if "wtags" in r:
                wtags.append(r["wtags"][w0:w1])
        T = len(ids[0]); assert all(len(x) == T for x in ids)
        Wm = max(1, max(len(x) for x in wpos))
        P = torch.zeros(len(b), Wm, dtype=torch.long); M = torch.zeros(len(b), Wm, dtype=torch.bool)
        for i, x in enumerate(wpos):
            P[i, :len(x)] = torch.tensor(x, dtype=torch.long); M[i, :len(x)] = True
        out = dict(ids=torch.tensor(ids, dtype=torch.long), wpos=P, wmask=M)
        if wtags:
            Tg = torch.full((len(b), Wm), nd.IGNORE, dtype=torch.long)
            for i, x in enumerate(wtags):
                Tg[i, :len(x)] = torch.tensor(x, dtype=torch.long)
            out["wtags"] = Tg
            if head == "span":
                S = torch.zeros(len(b), Wm, maxw, dtype=torch.long)
                for i, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
                    for gi, gj, l in docs[d]["wspans"]:
                        if gj < w0 or gi >= w1:
                            continue
                        if gi >= w0 and gj < w1:                    # entity inside the window
                            if gj - gi < maxw: S[i, gi - w0, gj - gi] = l
                        elif gi >= w0:                              # runs past the window end: unknown
                            S[i, gi - w0, :] = nd.IGNORE
                        else:                                       # started before the window: unknown
                            for a in range(max(w0, gj - maxw + 1), gj + 1):
                                S[i, a - w0, gj - a] = nd.IGNORE
                out["stgt"] = S
        return {k: v.to(device) for k, v in out.items()}, ranges


def length_batches(win, bs, shuffle, rng=None):
    by_len = {}
    for w in win:
        by_len.setdefault(w[2], []).append(w)
    out = []
    for _, ws in sorted(by_len.items()):
        if shuffle: rng.shuffle(ws)
        out += [ws[i:i + bs] for i in range(0, len(ws), bs)]
    if shuffle: rng.shuffle(out)
    return out


# ======================================================================================================
# 5. Prediction: windows -> document-level scores -> spans
# ======================================================================================================
@torch.no_grad()
def predict_scores(net, docs, wd, stride, bs, device):
    """returns per document: linear -> averaged tag probabilities (n_words,K); crf -> averaged emissions;
    span -> {(i,j): averaged class probabilities}. Overlapping windows are combined with a tent weight
    (a word in the middle of a window has more context than one at its edge)."""
    net.eval()
    model = net.student if isinstance(net, KDWrapper) else net
    head = model.head_type
    out = {}
    for b in length_batches(wd.windows(docs, stride), bs, shuffle=False):
        batch, ranges = wd.batch(docs, b, device)
        res = net(batch["ids"], batch["wpos"], batch["wmask"]).float().cpu()
        for k, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
            n = w1 - w0
            if n == 0:
                continue
            r = docs[d]
            pos = torch.tensor([r["wtok"][w] - s for w in range(w0, w1)], dtype=torch.float)
            wt = 1 + torch.minimum(pos, L - 1 - pos)
            if head == "span":
                cand = out.setdefault(d, {})
                pr = res[k, :n].softmax(-1)                                 # (n, maxw, C)
                keep = (pr[..., 1:].max(-1).values > 0.2).nonzero().tolist()     # plausible entities only
                for i, kk in keep:
                    j = i + kk
                    if j >= n: continue
                    key = (w0 + i, w0 + j); ww = float(min(wt[i], wt[j]))
                    acc = cand.get(key)
                    cand[key] = [pr[i, kk] * ww, ww] if acc is None else [acc[0] + pr[i, kk] * ww, acc[1] + ww]
            else:
                if d not in out:
                    nw = len(r["wtok"]); out[d] = [torch.zeros(nw, res.size(-1)), torch.zeros(nw, 1)]
                v = res[k, :n].softmax(-1) if head == "linear" else res[k, :n]
                out[d][0][w0:w1] += v * wt[:, None]; out[d][1][w0:w1] += wt[:, None]
    final = {}
    for d in docs:
        if head == "span":
            final[d] = {key: a / w for key, (a, w) in out.get(d, {}).items()}
        else:
            v = out.get(d)
            final[d] = (v[0] / v[1].clamp(min=1e-9)) if v is not None else torch.zeros(0, len(model.tags))
    return final


def decode(scores, docs, model, texts, strategy="viterbi", propagate=False):
    """document scores -> {doc: [(s, e, label)]}.
    linear head: "greedy" (argmax per word + lenient BIO repair) or "viterbi" (best BIO-valid sequence
    under the word log-probabilities); crf head: Viterbi with the learned transitions; span head: greedy
    non-overlapping spans by probability. propagate: document-level label consistency (see ner_data)."""
    head, tags = model.head_type, model.tags
    id2tag = dict(enumerate(tags))
    pred = {}
    if head != "span":
        if head == "crf":
            trans, start, end = [x.detach().float().cpu() for x in model.crf.scores()]
        else:
            ok, start_ok = nd.allowed_transitions(tags)
            trans = (~torch.tensor(ok)).float() * NEG; start = (~torch.tensor(start_ok)).float() * NEG
            end = torch.zeros(len(tags))
    for d, sc in scores.items():
        r = docs[d]
        if head == "span":
            cands = {k: (float(v[1:].max()), int(v.argmax())) for k, v in sc.items()}
            spans = nd.spans_from_candidates(cands, r["wcs"], r["wce"], model.id2label)
        elif head == "linear" and strategy == "greedy":
            spans = nd.spans_from_tags(sc.argmax(-1).tolist(), r["wcs"], r["wce"], id2tag)
        else:
            em = sc.clamp(min=1e-9).log() if head == "linear" else sc
            spans = nd.spans_from_tags(viterbi(em, trans, start, end), r["wcs"], r["wce"], id2tag)
        pred[d] = nd.propagate(spans, texts[d]) if propagate else spans
    return pred


def decoding_options(head):
    base = ["greedy", "viterbi"] if head == "linear" else ["viterbi"] if head == "crf" else ["greedy"]
    return [(s, p) for s in base for p in (False, True)]


# ======================================================================================================
# 6. Training
# ======================================================================================================
def train(net, train_docs, val_docs, val_texts, val_gold, labels, wd, cfg, device, log=print):
    """AdamW (backbone/adaptation LR + head LR), linear warm-up and decay, fp16, gradient accumulation,
    DataParallel over all GPUs; evaluates strict macro-F1 on the validation documents after every epoch
    and keeps the best weights (early stopping with patience)."""
    rng = random.Random(cfg["seed"])
    model = net.student if isinstance(net, KDWrapper) else net
    gpus = torch.cuda.device_count()
    para = nn.DataParallel(net) if gpus > 1 else net
    is_head = lambda n: n.startswith(("head.", "crf.", "student.head.", "student.crf."))
    named = [(n, p) for n, p in net.named_parameters() if p.requires_grad]
    groups = [dict(params=[p for n, p in named if not is_head(n)], lr=cfg["lr"]),
              dict(params=[p for n, p in named if is_head(n)], lr=cfg["lr_head"])]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    win = wd.windows(train_docs, cfg["train_stride"])
    acc = cfg.get("accum", 1)
    steps_per_epoch = math.ceil(len(win) / cfg["batch_size"] / acc)
    total = steps_per_epoch * cfg["epochs"]; warm = max(1, int(cfg["warmup_frac"] * total))
    log(f"train windows {len(win):,} of {wd.size} tokens (+{wd.H} header) | {steps_per_epoch} steps/epoch x "
        f"{cfg['epochs']} epochs | trainable {sum(p.numel() for p in params):,} | GPUs {max(gpus, 1)}")
    best, best_state, history, step, bad, t0 = -1.0, None, [], 0, 0, time.time()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    maxw = getattr(model, "maxw", None)
    for ep in range(1, cfg["epochs"] + 1):
        net.train(); run, n = 0.0, 0
        bl = length_batches(win, cfg["batch_size"], shuffle=True, rng=rng)
        for bi, b in enumerate(bl):
            batch, _ = wd.batch(train_docs, b, device, model.head_type, maxw)
            loss = para(**batch).mean() / acc
            scaler.scale(loss).backward()
            run += loss.item() * acc; n += 1
            if (bi + 1) % acc and bi + 1 < len(bl):
                continue
            step += 1
            f = step / warm if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if step % cfg["log_every"] == 0:
                log(f"  ep {ep} step {step}/{total} | loss {run / n:.4f} | {(time.time() - t0) / 60:.1f} min")
                run, n = 0.0, 0
        if cfg.get("select_last"):                    # refit on all documents: fixed epochs, no selection
            history.append(dict(epoch=ep, step=step, minutes=round((time.time() - t0) / 60, 1)))
            log(f"epoch {ep} done (refit on all documents: no validation-based selection)")
            continue
        sc = predict_scores(net, val_docs, wd, cfg["eval_stride"], cfg["eval_batch"], device)
        strategy = "viterbi" if model.head_type != "span" else "greedy"
        res = nd.strict_scores(decode(sc, val_docs, model, val_texts, strategy), val_gold, labels)
        history.append(dict(epoch=ep, step=step, macro_F1=res["macro_F1"], micro_F1=res["micro_F1"],
                            macro_P=res["macro_P"], macro_R=res["macro_R"], minutes=round((time.time() - t0) / 60, 1)))
        note = ""
        if res["macro_F1"] > best + 1e-4:
            best, bad, note = res["macro_F1"], 0, "  <- best"
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in net.state_dict().items()}
        else:
            bad += 1
        log(f"epoch {ep}: val strict macro-F1 {res['macro_F1']:.4f} | micro {res['micro_F1']:.4f} | "
            f"P {res['macro_P']:.3f} R {res['macro_R']:.3f}{note}")
        if cfg.get("patience") and bad >= cfg["patience"]:
            log(f"early stopping: no improvement for {bad} epochs"); break
        if (time.time() - t0) / 3600 > cfg["max_train_hours"]:
            log("time budget reached - stopping"); break
    if best_state is not None:
        net.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    return dict(best_val_macro_F1=best, history=history, train_minutes=round((time.time() - t0) / 60, 1),
                peak_gpu_gb=round(peak, 2))


def trainable_state(net):
    """only the trained tensors (what a PEFT method has to store per task)"""
    names = {n for n, p in net.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in net.state_dict().items() if k in names}


@torch.no_grad()
def self_test(net, wd, docs, device, head, maxw=None):
    """one forward + backward on two real windows before the long run (catches shape/device errors early)"""
    b = wd.windows(docs, wd.size)[:2]
    if len(b) == 2 and b[0][2] != b[1][2]: b = b[:1]
    batch, _ = wd.batch(docs, b, device, head, maxw)
    net.train()
    with torch.enable_grad():
        loss = net(**batch).mean(); loss.backward()
    net.zero_grad(set_to_none=True)
    out = net(batch["ids"], batch["wpos"], batch["wmask"])
    return float(loss), tuple(out.shape)

In [ ]:
%%writefile p2src/ner_run.py
# Phase 2 (L-NER) experiment runner: one call = one complete experiment
# (tokenise -> build model -> adapt -> self-test -> train -> choose decoding on validation -> analyse ->
#  predict the test fold -> submission zip + record). Shared by every Phase-2 notebook.
import gc
import json
import math
import os
import time
import zipfile
from collections import Counter

import torch

import ner_data as nd
import ner_model as nm

_DOC_CACHE = {}
SCORES = {}                      # run name -> word-level probabilities (val + test), filled when cfg["keep_scores"]


def _docs(D, arch, split, tag2id, label2id):
    key = (nm.ARCH_INFO[arch][0], split, tuple(D["train_ids"]), tuple(D["val_ids"]))
    if key not in _DOC_CACHE:
        tok = nm.load_tokenizer(arch, D["root"])
        tr, conf = nm.prepare_docs({d: D["texts"][d] for d in D["train_ids"]}, D["gold"], tok, tag2id, label2id, split)
        va, _ = nm.prepare_docs({d: D["texts"][d] for d in D["val_ids"]}, D["gold"], tok, tag2id, label2id, split)
        te, _ = nm.prepare_docs(D["test_texts"], None, tok, tag2id, label2id, split)
        _DOC_CACHE[key] = (tok, tr, va, te, conf)
    return _DOC_CACHE[key]


def build(cfg, D, tags, device):
    bb = nm.Backbone(cfg["arch"], D["root"], bidirectional=cfg["bidirectional"],
                     keep_layers=cfg.get("keep_layers"), init=cfg.get("init", "pretrained"))
    net = nm.NERModel(bb, tags, D["labels"], head=cfg["head"], dropout=cfg["dropout"],
                      class_weights=cfg.get("_class_weights"), max_span_words=cfg.get("_maxw", 32),
                      neg_weight=cfg.get("span_neg_weight", 0.5))
    desc = nm.apply_method(net, cfg)
    return net.to(device), desc


def run_experiment(cfg, D, device, out_root, log=print, teacher=None, keep_net=False):
    """cfg: one experiment's settings; D: data dict (texts, gold, test_texts, labels, train_ids, val_ids,
    conv, root). teacher: a trained NERModel -> knowledge distillation into the model built from cfg."""
    t_start = time.time()
    run_dir = os.path.join(out_root, cfg["run"]); os.makedirs(run_dir, exist_ok=True)
    labels = D["labels"]
    if cfg.get("refit_all"):    # final model: train on all 70 documents (validation scores below are then optimistic)
        D = dict(D, train_ids=sorted(set(D["train_ids"]) | set(D["val_ids"])))
        cfg = dict(cfg, select_last=True, patience=0)
        log(f"REFIT on all {len(D['train_ids'])} documents for {cfg['epochs']} epochs, decoding {cfg.get('decoding')} "
            "(validation documents are now part of training: their scores are NOT an estimate)")
    tags, tag2id = nd.tag_vocab(labels)
    label2id = {l: i + 1 for i, l in enumerate(labels)}
    tok, tr, va, te, conf = _docs(D, cfg["arch"], cfg["split"], tag2id, label2id)
    wd = nm.Windowing(tok, cfg["window"], cfg["header_tokens"], nm.ARCH_INFO[cfg["arch"]][1])
    cfg = dict(cfg)
    cfg["train_stride"] = cfg.get("train_stride") or wd.size // 2
    cfg["eval_stride"] = cfg.get("eval_stride") or (wd.size * 3) // 4
    widths = sorted(j - i + 1 for r in tr.values() for i, j, _ in r["wspans"])
    if cfg["head"] == "span":
        cfg["_maxw"] = cfg.get("max_span_words") or min(48, widths[int(0.995 * (len(widths) - 1))])
        log(f"span head: max {cfg['_maxw']} words per span covers "
            f"{sum(w <= cfg['_maxw'] for w in widths) / len(widths):.2%} of training entities")
    if cfg.get("class_weighting") == "sqrt_inv":
        c = Counter(t for r in tr.values() for t in r["wtags"])
        tot = sum(c.values())
        w = torch.tensor([math.sqrt(tot / max(1, c[i])) for i in range(len(tags))]); cfg["_class_weights"] = w / w[0]
    torch.manual_seed(cfg["seed"])
    net, desc = build(cfg, D, tags, device)
    total, trainable = nm.count_params(net)
    log(f"\n=== {cfg['run']} | {cfg['arch']} | {desc} | head {cfg['head']} | bidirectional {cfg['bidirectional']} | "
        f"params {total:,}, trainable {trainable:,} ({trainable / total:.2%})")
    model = net
    if teacher is not None:
        net = nm.KDWrapper(model, teacher, T=cfg["kd_T"], alpha=cfg["kd_alpha"]).to(device)
    loss0, shape = nm.self_test(net, wd, tr, device, cfg["head"], cfg.get("_maxw"))
    log(f"self-test ok: loss {loss0:.3f}, output {shape}")

    res = nm.train(net, tr, va, D["texts"], {d: D["gold"][d] for d in D["val_ids"]}, labels, wd, cfg, device, log)

    # ---- choose the decoding strategy on the validation documents ----
    t0 = time.time()
    vsc = nm.predict_scores(net, va, wd, cfg["eval_stride"], cfg["eval_batch"], device)
    infer_sec = time.time() - t0
    vgold = {d: D["gold"][d] for d in D["val_ids"]}
    options = {}
    opts = nm.decoding_options(cfg["head"])
    if cfg.get("decoding"):                           # fixed (chosen earlier on validation), e.g. for a refit
        s_, p_ = cfg["decoding"].split("+")[0], cfg["decoding"].endswith("+propagate")
        opts = [(s_, p_)]
    for strat, prop in opts:
        p = nm.decode(vsc, va, model, D["texts"], strat, prop)
        options[f"{strat}{'+propagate' if prop else ''}"] = (nd.strict_scores(p, vgold, labels), p, (strat, prop))
    best_name = max(options, key=lambda k: options[k][0]["macro_F1"])
    VAL, vpred, (strat, prop) = options[best_name]
    log("decoding on validation: " + " | ".join(f"{k} {v[0]['macro_F1']:.4f}" for k, v in options.items())
        + f"  -> {best_name}")
    log(f"VAL strict macro-F1 {VAL['macro_F1']:.4f} | P {VAL['macro_P']:.4f} | R {VAL['macro_R']:.4f} | "
        f"micro-F1 {VAL['micro_F1']:.4f}")
    log(f"{'label':<10} {'P':>6} {'R':>6} {'F1':>6} {'support':>8}")
    for l in labels:
        r = VAL["per_label"][l]; log(f"{l:<10} {r['P']:6.3f} {r['R']:6.3f} {r['F1']:6.3f} {r['support']:8d}")
    json.dump(dict(decoding=best_name, pred={d: [list(x) for x in v] for d, v in vpred.items()},
                   gold={d: [list(x) for x in v] for d, v in vgold.items()}),
              open(os.path.join(run_dir, "val_predictions.json"), "w"))
    ERR = nd.error_breakdown(vpred, vgold)
    CONF = nd.label_confusion(vpred, vgold).most_common(8)
    log(f"errors: {ERR}\nmost confused labels (gold -> predicted, exact span): {CONF}")

    # ---- test fold -> submission ----
    tsc = nm.predict_scores(net, te, wd, cfg["eval_stride"], cfg["eval_batch"], device)
    if cfg.get("keep_scores"):                        # for ensembles: word-level probabilities of this model
        SCORES[cfg["run"]] = dict(val=vsc, test=tsc, va=va, te=te, tags=tags, head=cfg["head"],
                                  tok=nm.ARCH_INFO[cfg["arch"]][0], split=cfg["split"])
    tpred = nm.decode(tsc, te, model, D["test_texts"], strat, prop)
    sub = nd.to_submission(tpred, D["conv"])
    assert set(sub) == set(D["test_texts"])
    for d, rows in sub.items():
        for s, e, l in rows:
            assert l in labels and 0 <= s < e <= len(D["test_texts"][d])
    pj = os.path.join(run_dir, "prediction.json"); json.dump(sub, open(pj, "w"), ensure_ascii=False)
    zp = os.path.join(run_dir, f"submission_{cfg['run']}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"] and set(json.loads(z.read("prediction.json"))) == set(sub)
    n_ent = sum(map(len, sub.values()))
    log(f"test: {n_ent:,} entities in {len(sub)} documents -> {zp}")

    # ---- save ----
    state = nm.trainable_state(model)
    torch.save(state, os.path.join(run_dir, "trainable_weights.pt"))
    trained_mb = sum(v.numel() * v.element_size() for v in state.values()) / 2**20
    if cfg.get("save_full"):
        torch.save({"model": model.state_dict(), "cfg": {k: v for k, v in cfg.items() if not k.startswith("_")},
                    "tags": tags}, os.path.join(run_dir, "ner_best.pt"))
    rec = dict(run=cfg["run"], arch=cfg["arch"], method=desc, head=cfg["head"], bidirectional=cfg["bidirectional"],
               kd=teacher is not None, cfg={k: v for k, v in cfg.items() if not k.startswith("_")},
               params_total=total, params_trainable=trainable, trainable_share=trainable / total,
               trained_weights_mb=round(trained_mb, 1), val_inference_sec=round(infer_sec, 1),
               decoding=best_name, decoding_options={k: round(v[0]["macro_F1"], 4) for k, v in options.items()},
               val=dict(macro_F1=VAL["macro_F1"], macro_P=VAL["macro_P"], macro_R=VAL["macro_R"],
                        micro_F1=VAL["micro_F1"], per_label=VAL["per_label"]),
               errors=ERR, confusion=[[a, b, n] for (a, b), n in CONF], test_entities=n_ent,
               test_label_counts=dict(Counter(l for rows in sub.values() for _, _, l in rows)),
               overlapping_gold_skipped=conf, zip=os.path.basename(zp), minutes_total=round((time.time() - t_start) / 60, 1),
               **res)
    json.dump(rec, open(os.path.join(run_dir, "record.json"), "w"), indent=1)
    log(f"saved {run_dir}/record.json, trainable_weights.pt ({trained_mb:.1f} MB) | {rec['minutes_total']} min total")
    if keep_net:
        return rec, model
    del net, model; gc.collect(); torch.cuda.empty_cache()
    return rec, None


def summary_table(records, log=print):
    log(f"\n{'run':<34} {'method':<34} {'head':<6} {'trainable':>11} {'val F1':>7} {'P':>6} {'R':>6} {'min':>5}")
    for r in records:
        log(f"{r['run']:<34} {r['method'][:34]:<34} {r['head']:<6} {r['params_trainable']:>11,} "
            f"{r['val']['macro_F1']:7.4f} {r['val']['macro_P']:6.3f} {r['val']['macro_R']:6.3f} {r['minutes_total']:5.1f}")


@torch.no_grad()
def evaluate_model(model, cfg, D, device, log=print):
    """validation macro-F1 and inference time of an already trained model (e.g. a loaded teacher)"""
    labels = D["labels"]
    tags, tag2id = nd.tag_vocab(labels)
    label2id = {l: i + 1 for i, l in enumerate(labels)}
    tok, tr, va, te, _ = _docs(D, cfg["arch"], cfg["split"], tag2id, label2id)
    wd = nm.Windowing(tok, cfg["window"], cfg["header_tokens"], nm.ARCH_INFO[cfg["arch"]][1])
    t0 = time.time()
    sc = nm.predict_scores(model, va, wd, cfg.get("eval_stride") or (wd.size * 3) // 4, cfg["eval_batch"], device)
    sec = time.time() - t0
    vgold = {d: D["gold"][d] for d in D["val_ids"]}
    best = max((nd.strict_scores(nm.decode(sc, va, model, D["texts"], s, p), vgold, labels)["macro_F1"], s, p)
               for s, p in nm.decoding_options(model.head_type))
    total, _ = nm.count_params(model)
    log(f"{cfg['run']}: val macro-F1 {best[0]:.4f} ({best[1]}{'+propagate' if best[2] else ''}) | params {total:,} | "
        f"val inference {sec:.1f}s")
    return dict(val_macro_F1=best[0], params_total=total, val_inference_sec=round(sec, 1))


def run_plan(plan, D, device, out_root, log=print, smoke=False):
    """plan: list of steps dict(cfg=..., keep_as=name or None, teacher=name or None).
    smoke=True : every step runs for 1 epoch; any error stops at once (that is the point of the check).
    smoke=False: a failing step is reported and the remaining steps still run."""
    import traceback
    store, records, failed = {}, [], []
    for step in plan:
        cfg = dict(step["cfg"])
        if smoke:
            cfg.update(run="smoke_" + cfg["run"], epochs=1, patience=0, save_full=False)
        teacher = store.get(step["teacher"]) if step.get("teacher") else None
        if step.get("teacher") and teacher is None:
            failed.append((cfg["run"], "teacher missing")); log(f"SKIP {cfg['run']}: teacher missing"); continue
        try:
            rec, net = run_experiment(cfg, D, device, out_root, log, teacher=teacher,
                                      keep_net=bool(step.get("keep_as")))
            records.append(rec)
            if step.get("keep_as"):
                net.eval(); store[step["keep_as"]] = net
        except Exception as e:
            if smoke:
                raise
            traceback.print_exc(); failed.append((cfg["run"], repr(e)))
        gc.collect(); torch.cuda.empty_cache()
    store.clear(); gc.collect(); torch.cuda.empty_cache()
    return records, failed


# ======================================================================================================
# Ensembles + training-data post-processing (selection on validation, then applied to refit models)
# ======================================================================================================
def ensemble_scores(runs, part):
    """average the word-level tag probabilities of several linear-head models (same tokenizer + split)"""
    first = SCORES[runs[0]]
    for r in runs[1:]:
        assert SCORES[r]["tok"] == first["tok"] and SCORES[r]["split"] == first["split"], "models must share words"
        assert SCORES[r]["head"] == "linear" == first["head"], "ensembles are built from linear-head models"
    return {d: sum(SCORES[r][part][d] for r in runs) / len(runs) for d in first[part]}


def decode_scores(sc, docs, tags, texts, propagate):
    import types
    return nm.decode(sc, docs, types.SimpleNamespace(head_type="linear", tags=tags), texts, "viterbi", propagate)


def select_on_val(names, D, post_model, log=print, max_members=None):
    """every subset of `names` (+/- propagation), then greedy forward selection of the post-processing rules,
    all scored by strict macro-F1 on the validation documents. Returns the chosen configuration."""
    import itertools
    import ner_post as npp
    first = SCORES[names[0]]
    va, tags = first["va"], first["tags"]
    vgold = {d: D["gold"][d] for d in D["val_ids"]}
    F = lambda p: nd.strict_scores(p, vgold, D["labels"])["macro_F1"]
    rows = []
    for k in range(1, (max_members or len(names)) + 1):
        for combo in itertools.combinations(names, k):
            sc = ensemble_scores(list(combo), "val")
            for prop in (False, True):
                p = decode_scores(sc, va, tags, D["texts"], prop)
                rows.append(dict(combo=list(combo), propagate=prop, F1=F(p), _pred=p))
    rows.sort(key=lambda r: -r["F1"])
    log("validation macro-F1 of model combinations (Viterbi decoding):")
    for r in rows[:12]:
        log(f"   {r['F1']:.4f}  {' + '.join(r['combo'])}{'  +propagate' if r['propagate'] else ''}")
    singles = {r["combo"][0]: r["F1"] for r in rows if len(r["combo"]) == 1 and not r["propagate"]}
    best = rows[0]
    rules, cur = [], best["F1"]
    log(f"post-processing on top of the best combination ({cur:.4f}):")
    for rule in npp.RULES:                                   # each rule alone
        p, info = npp.apply(best["_pred"], D["texts"], post_model, (rule,))
        log(f"   + {rule:<10} alone: {F(p):.4f}  {info}")
    remaining = list(npp.RULES)
    while remaining:                                          # greedy forward selection
        trial = {r: F(npp.apply(best["_pred"], D["texts"], post_model, tuple(rules + [r]))[0]) for r in remaining}
        r_best = max(trial, key=trial.get)
        if trial[r_best] <= cur + 1e-4:
            break
        rules.append(r_best); remaining.remove(r_best); cur = trial[r_best]
        log(f"   keep {r_best:<10} -> {cur:.4f}")
    choice = dict(combo=best["combo"], propagate=best["propagate"], rules=rules, val_F1_models=best["F1"],
                  val_F1_final=cur, singles=singles,
                  table=[dict(combo=r["combo"], propagate=r["propagate"], F1=r["F1"]) for r in rows])
    log(f"CHOSEN: {' + '.join(choice['combo'])} | propagate {choice['propagate']} | rules {rules} | val macro-F1 {cur:.4f}")
    return choice


def write_ensemble_zip(runs, propagate, rules, D, post_model, out_dir, name, log=print):
    import ner_post as npp
    first = SCORES[runs[0]]
    sc = ensemble_scores(list(runs), "test")
    pred = decode_scores(sc, first["te"], first["tags"], D["test_texts"], propagate)
    pred, info = npp.apply(pred, D["test_texts"], post_model, tuple(rules)) if rules else (pred, {})
    sub = nd.to_submission(pred, D["conv"])
    assert set(sub) == set(D["test_texts"])
    for d, rows in sub.items():
        for s, e, l in rows:
            assert l in D["labels"] and 0 <= s < e <= len(D["test_texts"][d])
    os.makedirs(os.path.join(out_dir, name), exist_ok=True)
    pj = os.path.join(out_dir, name, "prediction.json"); json.dump(sub, open(pj, "w"), ensure_ascii=False)
    zp = os.path.join(out_dir, name, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    log(f"{name}: {sum(map(len, sub.values())):,} entities {info} -> {zp}")
    return zp


@torch.no_grad()
def predict_texts(models, cfg, texts, D, device, propagate=True, log=print):
    """pseudo-labels for unlabelled documents: average the word probabilities of `models` (same tokenizer,
    linear heads), Viterbi + optional propagation. Returns {doc: [(s, e, label)]}."""
    import types
    tags, tag2id = nd.tag_vocab(D["labels"])
    tok = nm.load_tokenizer(cfg["arch"], D["root"])
    docs, _ = nm.prepare_docs(texts, None, tok, tag2id, None, cfg["split"])
    wd = nm.Windowing(tok, cfg["window"], cfg["header_tokens"], nm.ARCH_INFO[cfg["arch"]][1])
    stride = cfg.get("eval_stride") or (wd.size * 3) // 4
    t0 = time.time()
    scs = [nm.predict_scores(m, docs, wd, stride, cfg["eval_batch"], device) for m in models]
    sc = {d: sum(s[d] for s in scs) / len(scs) for d in docs}
    pred = nm.decode(sc, docs, types.SimpleNamespace(head_type="linear", tags=tags), texts, "viterbi", propagate)
    n = sum(map(len, pred.values()))
    log(f"pseudo-labelled {len(pred)} documents: {n:,} entities ({n / max(1, len(pred)):.0f} per document) in "
        f"{time.time() - t0:.0f}s | labels {dict(Counter(l for v in pred.values() for *_, l in v).most_common())}")
    return pred


# ======================================================================================================
# Per-label decision biases for macro-F1 (tuned on out-of-fold predictions)
# ======================================================================================================
def viterbi_batch(ems, trans, start, end):
    """BIO-constrained Viterbi for many documents at once (CPU). ems: list of (n_i, K) score tensors."""
    B, K = len(ems), trans.size(0)
    N = max([e.size(0) for e in ems] + [1])
    E = torch.zeros(B, N, K); M = torch.zeros(B, N, dtype=torch.bool)
    for i, e in enumerate(ems):
        E[i, :e.size(0)] = e; M[i, :e.size(0)] = True
    score = start[None] + E[:, 0]
    back = torch.zeros(B, N, K, dtype=torch.int32)
    ident = torch.arange(K, dtype=torch.int32)[None].expand(B, K)
    for t in range(1, N):
        best, idx = (score[:, :, None] + trans[None]).max(1)
        m = M[:, t, None]
        score = torch.where(m, best + E[:, t], score)
        back[:, t] = torch.where(m, idx.to(torch.int32), ident)       # padding: keep the tag
    tag = (score + end[None]).argmax(1)
    out = torch.zeros(B, N, dtype=torch.long)
    for t in range(N - 1, 0, -1):
        out[:, t] = tag
        tag = back[:, t].long().gather(1, tag[:, None]).squeeze(1)
    out[:, 0] = tag
    return [out[i, :e.size(0)].tolist() for i, e in enumerate(ems)]


def decode_bias(sc, docs, tags, texts, bias, propagate=True):
    """add bias[label] to the log-probability of B-label and I-label, then BIO Viterbi (+ propagation)"""
    ok, start_ok = nd.allowed_transitions(tags)
    trans = (~torch.tensor(ok)).float() * nm.NEG
    start = (~torch.tensor(start_ok)).float() * nm.NEG
    end = torch.zeros(len(tags))
    bv = torch.tensor([0.0 if t == "O" else float(bias.get(t[2:], 0.0)) for t in tags])
    ids = list(sc)
    paths = viterbi_batch([sc[d].clamp(min=1e-9).log() + bv for d in ids], trans, start, end)
    id2tag = dict(enumerate(tags))
    pred = {}
    for d, p in zip(ids, paths):
        r = docs[d]
        spans = nd.spans_from_tags(p, r["wcs"], r["wce"], id2tag)
        pred[d] = nd.propagate(spans, texts[d]) if propagate else spans
    return pred


def tune_biases(sc, docs, tags, texts, gold, labels, grid=(-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0),
                passes=2, log=print):
    """coordinate ascent: one bias per label, each pass tries every grid value for every label"""
    bias = {l: 0.0 for l in labels}
    F = lambda b: nd.strict_scores(decode_bias(sc, docs, tags, texts, b), gold, labels)["macro_F1"]
    base = best = F(bias)
    for p in range(passes):
        for l in labels:
            trial = {g: (best if g == bias[l] else F(dict(bias, **{l: g}))) for g in grid}
            g = max(trial, key=trial.get)
            if trial[g] > best + 1e-4:
                bias[l], best = g, trial[g]
        log(f"   pass {p + 1}: macro-F1 {best:.4f} (no bias {base:.4f}) | biases { {k: v for k, v in bias.items() if v} }")
    return bias, base, best

In [ ]:
%%writefile p2src/ner_post.py
# Phase 2 (L-NER) post-processing learned ONLY from the training annotations (pure Python, no model):
#   1. precision filter  - drop a predicted (string, label) that the training annotators almost never label
#                          (e.g. the bare word "Tribunal" predicted as AUTH)
#   2. gazetteer         - add (string, label) pairs that are (almost) always annotated in the training documents
#   3. year rule         - STAT: include / exclude a following ", 1996" as the training annotations do
#   4. citation rule     - PREC: include / exclude a following law-report citation "(2004) 9 SCC 27", "AIR 1962 SC 605"
# Each rule is switched on only if it improves strict macro-F1 on the validation documents.
import bisect
import re
from collections import Counter, defaultdict


def _word_boundary(t, i, j):
    return (i == 0 or not t[i - 1].isalnum()) and (j == len(t) or not t[j].isalnum())


class TrainStats:
    """how often each string occurs in the training documents, and how often it is annotated with each label"""

    def __init__(self, texts, gold):
        self.texts = [texts[d] for d in gold]
        self.gold = Counter((texts[d][s:e], l) for d in gold for s, e, l in gold[d])
        self.by_string = defaultdict(Counter)
        for (s, l), n in self.gold.items():
            self.by_string[s][l] += n
        self._occ = {}

    def occ(self, s):
        """whole-word occurrences of s in the training documents"""
        if s not in self._occ:
            n = 0
            for t in self.texts:
                i = t.find(s)
                while i >= 0:
                    n += _word_boundary(t, i, i + len(s))
                    i = t.find(s, i + 1)
            self._occ[s] = n
        return self._occ[s]


def precision_filter(pred, texts, st, min_occ=5, max_ratio=0.15):
    out, dropped = {}, Counter()
    for d, spans in pred.items():
        keep = []
        for s, e, l in spans:
            t = texts[d][s:e]
            o = st.occ(t)
            if o >= min_occ and st.gold[(t, l)] / o < max_ratio:
                dropped[(t, l)] += 1
                continue
            keep.append((s, e, l))
        out[d] = keep
    return out, dropped


def gazetteer_entries(st, min_count=3, min_ratio=0.8, min_chars=4):
    out = []
    for s, labs in st.by_string.items():
        l, n = labs.most_common(1)[0]
        if n >= min_count and len(s) >= min_chars and re.search(r"[A-Za-z]", s):
            o = st.occ(s)
            if o and n / o >= min_ratio:
                out.append((s, l))
    return sorted(out, key=lambda x: -len(x[0]))          # longest first


def apply_gazetteer(pred, texts, entries):
    out, added = {}, 0
    for d, spans in pred.items():
        t = texts[d]
        cov = sorted((s, e) for s, e, _ in spans)
        starts = [s for s, _ in cov]
        new = list(spans)
        for string, l in entries:
            i = t.find(string)
            while i >= 0:
                j = i + len(string)
                if _word_boundary(t, i, j):
                    k = bisect.bisect_left(starts, j)
                    if not any(cs < j and ce > i for cs, ce in cov[max(0, k - 30):k]):
                        new.append((i, j, l)); added += 1
                        k2 = bisect.bisect_left(starts, i); cov.insert(k2, (i, j)); starts.insert(k2, i)
                i = t.find(string, i + 1)
        out[d] = sorted(new)
    return out, added


# ---- boundary rules: learn from the training annotations whether a tail is part of the entity ----------------
YEAR_TAIL = re.compile(r",?\s?\d{4}\b")
CITE_TAIL = re.compile(r",?\s?(?:\(\d{4}\)|\[\d{4}\])\s?\d{0,3}\s?[A-Z][A-Za-z.]{0,12}\s?(?:\(\w+\)\s?)?\d{1,5}"
                       r"|,?\s?AIR\s?\d{4}\s?[A-Z][A-Za-z.]{0,12}\s?\d{1,5}")


def learn_tail(texts, gold, label, tail_re):
    """P(the gold entity includes the tail | such a tail exists right after the entity core)"""
    inc = exc = 0
    for d, spans in gold.items():
        t = texts[d]
        for s, e, l in spans:
            if l != label:
                continue
            if tail_re.match(t, e):
                exc += 1                                     # tail follows, annotators left it out
            else:
                txt = t[s:e]
                for m in tail_re.finditer(txt):
                    if m.end() == len(txt) and m.start() > 0:
                        inc += 1; break                      # entity ends with such a tail
    return inc / max(1, inc + exc), inc, exc


def apply_tail(pred, texts, label, tail_re, include):
    out, changed = {}, 0
    for d, spans in pred.items():
        t = texts[d]
        spans = sorted(spans)
        nxt_start = [s for s, _, _ in spans[1:]] + [len(t) + 1]
        new = []
        for (s, e, l), ns in zip(spans, nxt_start):
            if l == label:
                if include:
                    m = tail_re.match(t, e)
                    if m and m.end() <= ns:
                        e = m.end(); changed += 1
                else:
                    txt = t[s:e]
                    for m in tail_re.finditer(txt):
                        if m.end() == len(txt) and m.start() >= 4:
                            e2 = s + m.start()
                            while e2 > s and not t[e2 - 1].isalnum() and t[e2 - 1] not in ").":
                                e2 -= 1
                            if e2 - s >= 4:
                                e = e2; changed += 1
                            break
            new.append((s, e, l))
        out[d] = new
    return out, changed


TITLE_ROLES = ("APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE")


def consistency(pred, texts, title_chars=2500):
    """Document-level label consistency (cf. the "reconciliation" step of Kalamkar et al. 2022): every predicted
    occurrence of the same string gets ONE label - the party/counsel/judge label it has in the cause title (first
    `title_chars` characters) if any, otherwise the majority label; ties without a title label are left alone."""
    out, changed = {}, 0
    for d, spans in pred.items():
        t = texts[d]
        groups = defaultdict(list)
        for s, e, l in spans:
            groups[t[s:e]].append((s, e, l))
        new = []
        for items in groups.values():
            labs = Counter(l for _, _, l in items)
            if len(labs) == 1:
                new += items; continue
            title = Counter(l for s, _, l in items if s < title_chars and l in TITLE_ROLES)
            if title:
                lab = title.most_common(1)[0][0]
            else:
                (lab, n1), (_, n2) = labs.most_common(2)
                if n1 == n2:
                    new += items; continue
            changed += sum(l != lab for _, _, l in items)
            new += [(s, e, lab) for s, e, _ in items]
        out[d] = sorted(new)
    return out, changed


RULES = ("filter", "gazetteer", "stat_year", "prec_cite", "consistency")


def fit(texts, gold):
    """everything the rules need, learned from the training documents"""
    st = TrainStats(texts, gold)
    p_year = learn_tail(texts, gold, "STAT", YEAR_TAIL)
    p_cite = learn_tail(texts, gold, "PREC", CITE_TAIL)
    return dict(stats=st, gazetteer=gazetteer_entries(st), stat_year=p_year, prec_cite=p_cite)


def apply(pred, texts, model, rules):
    log = {}                                # order: repair boundaries first, then filter, then add
    if "stat_year" in rules:
        pred, c = apply_tail(pred, texts, "STAT", YEAR_TAIL, model["stat_year"][0] >= 0.5); log["stat_year_changed"] = c
    if "prec_cite" in rules:
        pred, c = apply_tail(pred, texts, "PREC", CITE_TAIL, model["prec_cite"][0] >= 0.5); log["prec_cite_changed"] = c
    if "consistency" in rules:
        pred, c = consistency(pred, texts); log["consistency_relabelled"] = c
    if "filter" in rules:
        pred, dr = precision_filter(pred, texts, model["stats"]); log["filter_dropped"] = sum(dr.values())
    if "gazetteer" in rules:
        pred, a = apply_gazetteer(pred, texts, model["gazetteer"]); log["gazetteer_added"] = a
    return pred, log

In [ ]:
import importlib
sys.path.insert(0, os.path.abspath("p2src"))
import ner_data as nd, ner_model as nm, ner_run as nr
for _m in (nd, nm, nr): importlib.reload(_m)
print("Phase-2 code imported")

In [ ]:
# 4 · Load Phase_2_LNER, check the offset convention, document-level validation split
from collections import Counter
from huggingface_hub import HfApi, hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
files = [f for f in HfApi(token=HF_TOKEN).list_repo_files(REPO, repo_type="dataset") if f.startswith("Phase_2")]
get = lambda n: hf_hub_download(REPO, next(f for f in files if f.endswith(n)), repo_type="dataset", token=HF_TOKEN)
load = lambda n: json.load(open(get(n), encoding="utf-8"))
LABELS = [l.strip() for l in open(get("labels.txt"), encoding="utf-8") if l.strip()]
texts, gold_raw = {}, {}
for fold in ("fold1", "fold2"):
    t = nd.parse_texts(load(f"{fold}-text.json")); g = nd.parse_gold(load(f"{fold}-gold.json"))
    assert set(g) <= set(t)
    texts.update(t); gold_raw.update({d: g.get(d, []) for d in t})
test_texts = nd.parse_texts(load("fold3-text.json"))
CONV, info = nd.end_convention(texts, gold_raw)
GOLD = nd.to_exclusive(gold_raw, CONV)
assert set(l for v in GOLD.values() for *_, l in v) <= set(LABELS)
ids = sorted(texts); random.Random(1337).shuffle(ids)
VAL_IDS, TRAIN_IDS = sorted(ids[:10]), sorted(ids[10:])            # the same split in every Phase-2 notebook
D = dict(texts=texts, gold=GOLD, test_texts=test_texts, labels=LABELS, train_ids=TRAIN_IDS, val_ids=VAL_IDS,
         conv=CONV, root=ROOT)
D_SMOKE = dict(D, train_ids=TRAIN_IDS[:8], val_ids=VAL_IDS[:3])
print(f"{len(texts)} training documents ({len(TRAIN_IDS)} train / {len(VAL_IDS)} validation) | {len(test_texts)} test | "
      f"end offsets {CONV} ({max(info['exclusive_match'], info['inclusive_match'])}/{info['total_with_text']} match)")
print("entities per label (train):", dict(Counter(l for d in TRAIN_IDS for *_, l in GOLD[d])))

In [ ]:
# 5 · Offset round-trip check per tokenizer: gold spans -> word tags -> spans (best possible strict score)
tags, tag2id = nd.tag_vocab(LABELS); label2id = {l: i + 1 for i, l in enumerate(LABELS)}; id2tag = dict(enumerate(tags))
for arch in sorted({s["cfg"]["arch"] for s in PLAN}):
    tok = nm.load_tokenizer(arch, ROOT)
    for split in ("bpe", "fine"):
        rec = {}
        for d in TRAIN_IDS:
            ids_, wtok, wcs, wce = nd.encode_words(tok, texts[d], split)
            assert tok.decode(ids_) == texts[d]
            wt, _, _ = nd.word_targets(wcs, wce, GOLD[d], tag2id, label2id)
            rec[d] = nd.spans_from_tags(wt, wcs, wce, id2tag)
        sc = nd.strict_scores(rec, {d: GOLD[d] for d in TRAIN_IDS}, LABELS)
        print(f"{arch} ({nm.ARCH_INFO[arch][0]}) split={split:<4}: ceiling strict macro-F1 {sc['macro_F1']:.4f}")

In [ ]:
# 6 · Real runs on all 70 documents (smoke check skipped: identical code to W1, already proven)
t0 = time.time()
records, failed = nr.run_plan(PLAN, D, DEVICE, OUT)
print(f"finished in {(time.time() - t0) / 60:.1f} min" + (f" | FAILED {failed}" if failed else ""))

In [ ]:
# 7 · Save the test probabilities (input of notebook W4)
names = [s["cfg"]["run"] for s in PLAN if s["cfg"]["run"] in nr.SCORES]
torch.save({n: dict(test=nr.SCORES[n]["test"], arch="arch4_mla") for n in names}, os.path.join(OUT, f"test_scores_{TAG}.pt"))
print("saved", f"test_scores_{TAG}.pt", names)
nr.summary_table(records)